In [2]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import accuracy_score

# Load Data
df = pd.read_csv('datasets/changed/data_news_1.csv')



# Label Encoding (0 = Fake, 1 = Real)
df['label'] = df['type_of_news'].map({'Fake': 0, 'Real': 1})

# BERT Tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

class NewsDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = tokenizer(text, truncation=True, padding="max_length", max_length=64, return_tensors="pt")
        # Flatten the encoding dictionary and add the label to it
        return {**{key: val.squeeze(0) for key, val in encoding.items()}, 'label': torch.tensor(label)}

# **Train-Test Split**
X_train, X_test, y_train, y_test = train_test_split(df['title'], df['label'], test_size=0.2, random_state=42, stratify=df['label'])

train_dataset = NewsDataset(list(X_train), list(y_train))
test_dataset = NewsDataset(list(X_test), list(y_test))

# Load Pre-trained BERT Model
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

# **Accuracy Metric Function**
def compute_accuracy(p):
    preds, labels = p
    preds = np.argmax(preds, axis=1)  # Convert logits to predicted class labels
    accuracy = accuracy_score(labels, preds)
    return {"accuracy": accuracy}

# **Training Arguments**
training_args = TrainingArguments(
    output_dir="./bert_fake_news",
    eval_strategy="epoch",  # Changed to the new argument name
    save_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_accuracy,  # Pass the accuracy function here
)

# **Train Model**
trainer.train()

# **Evaluate Model**
results = trainer.evaluate()
print(f"Evaluation Results: {results}")

# **User Input Prediction**
def predict_news(title):
    model.eval()  # Make sure the model is in evaluation mode
    inputs = tokenizer(title, return_tensors="pt", truncation=True, padding="max_length", max_length=64)
    # Ensure inputs are on the same device as the model
    inputs = {key: val.to(model.device) for key, val in inputs.items()}
    with torch.no_grad():  # Disable gradient computation for inference
        outputs = model(**inputs)
    probs = F.softmax(outputs.logits, dim=1)
    pred = torch.argmax(probs, dim=1).item()
    return "Real" if pred == 1 else "Fake"

# Get user input and predict the news type
user_title = input("Enter news title: ")
print("Predicted News Type:", predict_news(user_title))


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.625188,0.648649
2,No log,0.557721,0.783784
3,No log,0.541551,0.756757


Evaluation Results: {'eval_loss': 0.5415514707565308, 'eval_accuracy': 0.7567567567567568, 'eval_runtime': 0.4004, 'eval_samples_per_second': 92.403, 'eval_steps_per_second': 12.487, 'epoch': 3.0}


Enter news title:  What is the largest goal ever


Predicted News Type: Fake
